# Delta Lake — Incremental Load & SCD using MERGE
**Week 7 Assignment — Celebal Technologies Data Engineering Internship**

The idea here is to take a customer master dataset, clean it up, load it into a Delta table, and then
apply a second "incremental" file to it using Delta Lake's `MERGE` operation — updating whatever
already exists and inserting whatever is new. This is the standard pattern for handling Slowly Changing
Dimensions (SCD) in a warehouse/lakehouse setting.

I've implemented two versions of the merge so the difference is actually visible:
- **SCD Type 1** — old value is simply overwritten. No history kept.
- **SCD Type 2** — old row is marked as expired (`is_current = false`) and a new row is inserted for
  the updated version, so the full history of changes is preserved.

Run top to bottom in a Databricks notebook (Delta Lake is built into the Databricks runtime, so no
extra setup is needed there).

In [ ]:
# STEP 1 — Load dataset into a Delta table
df_master_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/dataset/customer_master.csv")
)

display(df_master_raw)

In [ ]:
print("Rows in raw master file:", df_master_raw.count())

Before saving anything, worth pointing out what's actually in this file — there are 12 rows but
only 11 distinct `customer_id` values (id 2, Anjali Verma, shows up twice), and customer 11
(Sanjay Patel) has a missing email. Both of these get handled in the cleaning step below.

In [ ]:
# Save the raw data as a Delta table so we can run MERGE on it later
(
    df_master_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("customer_master_delta")
)

display(spark.table("customer_master_delta"))

## Step 2 — Data Cleaning

Two things need fixing: the duplicate `customer_id` row, and the null email. `dropDuplicates` on
`customer_id` takes care of the first, `dropna` on the `email` column takes care of the second.

In [ ]:
from pyspark.sql.functions import col

df_clean = (
    spark.table("customer_master_delta")
    .dropDuplicates(["customer_id"])
    .dropna(subset=["email"])
)

print("Rows before cleaning:", df_master_raw.count())
print("Rows after cleaning:", df_clean.count())

df_clean.orderBy("customer_id").display()

In [ ]:
# Overwrite the Delta table with the cleaned version
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("customer_master_delta")
)

print("customer_master_delta now has", spark.table("customer_master_delta").count(), "rows")

## Step 3 — Load the incremental file

This simulates a second day's worth of data arriving — some existing customers changed city/status/email,
and a couple of brand-new customers showed up.

In [ ]:
df_incremental = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/dataset/customer_incremental.csv")
)

print("Rows in incremental file:", df_incremental.count())
display(df_incremental)

## Step 4a — MERGE: SCD Type 1 (overwrite in place)

I'll build this one into its own table (`customer_scd1`) starting from the cleaned master data,
so it doesn't interfere with the SCD2 version below.

`whenMatchedUpdateAll()` overwrites every column of a matching row with the incoming values.
`whenNotMatchedInsertAll()` inserts anything that isn't already there.

In [ ]:
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("customer_scd1")
)

from delta.tables import DeltaTable

scd1_table = DeltaTable.forName(spark, "customer_scd1")

(
    scd1_table.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

display(spark.table("customer_scd1").orderBy("customer_id"))

Customers 1, 3, 6 and 8 got their rows updated in place (city/status/email changed — customer 8's
incremental row was actually identical to what was already there, so it just got rewritten with the
same values). Customers 11 and 12 are new inserts. Note customer 11 originally had a null email and got
dropped during cleaning — so as far as this table is concerned, "Meera Iyer" (id 11) looks like a brand
new customer rather than an update to the old "Sanjay Patel" record. Worth flagging in a real pipeline,
since the same ID was reused for a different person.

## Step 4b — MERGE: SCD Type 2 (keep history)

SCD1 is simple but it throws away the old value the moment something changes. SCD2 keeps it — the old
row gets marked as expired instead of being overwritten, and a new row is inserted as the current version.

This needs three extra columns: `effective_start_date`, `effective_end_date`, and `is_current`. The merge
itself happens in two passes: first expire the rows that actually changed, then insert the new versions
(plus any brand-new customers).

In [ ]:
from pyspark.sql.functions import lit, current_date

scd2_init = (
    df_clean
    .withColumn("effective_start_date", col("updated_at"))
    .withColumn("effective_end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

(
    scd2_init.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("customer_scd2")
)

display(spark.table("customer_scd2"))

In [ ]:
scd2_table = DeltaTable.forName(spark, "customer_scd2")

# Pass 1 - expire the current row for any customer whose data actually changed
change_condition = (
    "target.is_current = true AND ("
    "target.name <> source.name OR "
    "target.email <> source.email OR "
    "target.city <> source.city OR "
    "target.status <> source.status)"
)

(
    scd2_table.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.customer_id = source.customer_id AND target.is_current = true"
    )
    .whenMatchedUpdate(
        condition=change_condition,
        set={
            "is_current": "false",
            "effective_end_date": "source.updated_at"
        }
    )
    .execute()
)

print("After Pass 1 (expiring changed rows):")
display(spark.table("customer_scd2").orderBy("customer_id", "is_current"))

In [ ]:
# Pass 2 — insert a new "current" row for every customer that just got expired,
# plus any customer_id that wasn't in the table at all (brand-new customers)
expired_ids = (
    spark.table("customer_scd2")
    .filter("is_current = false")
    .select("customer_id")
)

existing_ids = df_clean.select("customer_id")

new_or_changed_ids = expired_ids.unionByName(
    df_incremental.join(existing_ids, on="customer_id", how="left_anti").select("customer_id")
).distinct()

rows_to_insert = (
    df_incremental.join(new_or_changed_ids, on="customer_id", how="inner")
    .withColumn("effective_start_date", col("updated_at"))
    .withColumn("effective_end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

rows_to_insert.write.format("delta").mode("append").saveAsTable("customer_scd2")

print("customer_scd2 now has", spark.table("customer_scd2").count(), "rows (includes history)")
display(spark.table("customer_scd2").orderBy("customer_id", "is_current"))

Ravi Sharma (id 1), Suresh Kumar (id 3) and Neha Gupta (id 6) each now have two rows — one expired,
one current — so their full change history is sitting right there in the table. Customer 8's row didn't
change (same data came in on the incremental file), so it correctly stayed as a single current row instead
of getting a pointless second version. Customers 11 and 12 came in as fresh current rows.

## Step 5 — Validation

Two checks: row counts make sense, and there are no duplicate *current* customer_id values (a customer
should never have two rows both marked `is_current = true` at the same time — that would mean the merge
logic is broken).

In [ ]:
print("=== Row counts ===")
print("customer_scd1:", spark.table("customer_scd1").count())
print("customer_scd2 (all rows, incl. history):", spark.table("customer_scd2").count())
print("customer_scd2 (current rows only):", spark.table("customer_scd2").filter("is_current = true").count())

In [ ]:
from pyspark.sql.functions import count as _count

print("=== Duplicate check: SCD1 ===")
dup_scd1 = spark.table("customer_scd1").groupBy("customer_id").count().filter("count > 1")
if dup_scd1.count() == 0:
    print("No duplicate customer_id values in customer_scd1.")
else:
    dup_scd1.show()

print("=== Duplicate check: SCD2 (current rows only) ===")
dup_scd2 = (
    spark.table("customer_scd2")
    .filter("is_current = true")
    .groupBy("customer_id").count().filter("count > 1")
)
if dup_scd2.count() == 0:
    print("No customer has more than one 'current' row in customer_scd2.")
else:
    dup_scd2.show()

## Step 6 — Final Output

In [ ]:
print("Final SCD1 table (overwrite-in-place):")
display(spark.table("customer_scd1").orderBy("customer_id"))

print("Final SCD2 table (full history):")
display(
    spark.table("customer_scd2")
    .orderBy("customer_id", "effective_start_date")
    .select("customer_id", "name", "city", "status",
            "effective_start_date", "effective_end_date", "is_current")
)

## Summary

- **Load** — `customer_master.csv` was read and saved as a Delta table.
- **Clean** — one duplicate `customer_id` row and one row with a missing email were removed.
- **Incremental load** — `customer_incremental.csv` was read in as the "new data arriving" file.
- **SCD1 merge** — matching customers were overwritten in place; new customers were inserted.
  Final table: 12 rows, no duplicate IDs.
- **SCD2 merge** — matching customers that actually changed got a new versioned row instead of
  being overwritten, so the old value is still there with `is_current = false`. Final table: 15 rows
  (10 original + 3 new versions for changed customers + 2 new customers), with exactly one `is_current = true`
  row per customer.
- **Validation** — row counts and duplicate checks confirmed both merges worked as expected.

## Conclusion

Delta Lake's `MERGE` turns what would otherwise be a manual "figure out what changed, delete, re-insert"
job into a single atomic statement. SCD1 is the simpler version and is fine when you only ever care about
the current value. SCD2 costs a bit more complexity (extra columns, a two-pass merge) but is what you'd
actually want in a real warehouse if you ever need to answer "what did this customer's city used to be
before March?" — which SCD1 can't answer at all, since it throws the old value away.